In [1]:
!pip install fastapi uvicorn nest-asyncio httpx psutil -q

import nest_asyncio, uvicorn, threading, time, uuid, psutil
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from datetime import datetime
from enum import Enum

nest_asyncio.apply()
app = FastAPI()

# ---- Request log with ID lookup ----
class RequestType(str, Enum):
    login = "login"
    logout = "logout"
    create = "create"
    update = "update"
    delete = "delete"

class RequestLog(BaseModel):
    client_id: str
    request_type: RequestType
    timestamp: datetime
    seconds: int

logs_db = {}  # id -> RequestLog

@app.post("/log")
def capture_request(entry: RequestLog):
    log_id = str(uuid.uuid4())
    logs_db[log_id] = entry
    return {"id": log_id, "status": "captured"}

@app.get("/log/{log_id}")
def get_log_by_id(log_id: str):
    if log_id not in logs_db:
        raise HTTPException(status_code=404, detail="Log not found")
    return {"id": log_id, **logs_db[log_id].dict()}

@app.get("/logs/past")
def get_past_logs():
    return [{"id": k, **v.dict()} for k, v in logs_db.items()]

# ---- System stats ----
@app.get("/system/stats")
def system_stats():
    return {
        "cpu_percent": psutil.cpu_percent(interval=1),
        "memory_percent": psutil.virtual_memory().percent,
        "memory_available_mb": round(psutil.virtual_memory().available / (1024**2), 2)
    }

@app.on_event("startup")
def on_startup():
    print("Server starting. CPU:", psutil.cpu_percent(interval=1), "% | Memory:", psutil.virtual_memory().percent, "%")

@app.on_event("shutdown")
def on_shutdown():
    print("Server stopping. CPU:", psutil.cpu_percent(interval=1), "% | Memory:", psutil.virtual_memory().percent, "%")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)
print("Server is running")

/tmp/ipykernel_49/2334484926.py:53: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")
/tmp/ipykernel_49/2334484926.py:57: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("shutdown")


Server starting. CPU: 27.2 % | Memory: 4.2 %
Server is running


In [2]:
import httpx
from datetime import datetime

# Login request -> get its ID back
login_payload = {
    "client_id": "user123",
    "request_type": "login",
    "timestamp": datetime.now().isoformat(),
    "seconds": 5
}
r1 = httpx.post("http://127.0.0.1:8000/log", json=login_payload)
print("POST /log:", r1.json())

log_id = r1.json()["id"]

# Fetch that specific login by ID
r2 = httpx.get(f"http://127.0.0.1:8000/log/{log_id}")
print("GET /log/{id}:", r2.json())

# System stats
r3 = httpx.get("http://127.0.0.1:8000/system/stats")
print("GET /system/stats:", r3.json())

POST /log: {'id': 'e441bb0e-538f-4ca1-9e31-e54a8b6fb604', 'status': 'captured'}
GET /log/{id}: {'id': 'e441bb0e-538f-4ca1-9e31-e54a8b6fb604', 'client_id': 'user123', 'request_type': 'login', 'timestamp': '2026-10-16T06:10:44.359924', 'seconds': 5}


/tmp/ipykernel_49/2334484926.py:38: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  return {"id": log_id, **logs_db[log_id].dict()}


GET /system/stats: {'cpu_percent': 0.5, 'memory_percent': 4.2, 'memory_available_mb': 30745.11}


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install fastapi uvicorn nest-asyncio httpx -q

import nest_asyncio, uvicorn, threading, time
from fastapi import FastAPI

nest_asyncio.apply()
app = FastAPI()

@app.get("/ping")
def ping():
    return {"message": "pong"}

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)
print("Server is running")

Server is running


In [4]:
!pip install fastapi uvicorn nest-asyncio httpx python-multipart -q

import nest_asyncio, uvicorn, threading, time
from fastapi import FastAPI, Form
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
from datetime import datetime
from enum import Enum

nest_asyncio.apply()
app = FastAPI()

# ---- JSON request log (Pydantic) ----
class RequestType(str, Enum):
    login = "login"
    logout = "logout"
    create = "create"
    update = "update"
    delete = "delete"

class RequestLog(BaseModel):
    client_id: str
    request_type: RequestType
    timestamp: datetime
    seconds: int

logs = []

@app.post("/log")
def capture_request(entry: RequestLog):
    logs.append(entry)
    return {"status": "captured"}

@app.get("/logs/past")
def get_past_logs():
    return logs  # FastAPI auto-serializes this to JSON

# ---- HTML form ----
responses = []

@app.get("/survey", response_class=HTMLResponse)
def show_form():
    return """
    <html><body>
        <h2>Survey</h2>
        <form action="/survey" method="post">
            Name: <input type="text" name="name"><br>
            Email: <input type="text" name="email"><br>
            Rating (1-5): <input type="number" name="rating" min="1" max="5"><br>
            <input type="submit" value="Submit">
        </form>
    </body></html>
    """

@app.post("/survey")
def submit_form(name: str = Form(...), email: str = Form(...), rating: int = Form(...)):
    entry = {"name": name, "email": email, "rating": rating}
    responses.append(entry)
    return {"status": "saved", "entry": entry}

@app.get("/survey/responses")
def get_responses():
    return responses

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8080, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)
print("Server is running")

ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


Server is running


In [6]:
import httpx
from datetime import datetime

# 1. JSON request log
log_payload = {
    "client_id": "user123",
    "request_type": "login",
    "timestamp": datetime.now().isoformat(),
    "seconds": 5
}
r1 = httpx.post("http://127.0.0.1:8080/log", json=log_payload)
print("POST /log:", r1.status_code, r1.json())

r2 = httpx.get("http://127.0.0.1:8080/logs/past")
print("GET /logs/past:", r2.json())

# 2. HTML form page
r3 = httpx.get("http://127.0.0.1:8080/survey")
print("GET /survey (HTML):", r3.text[:100], "...")

# 3. Submit the form (form-encoded, not JSON)
r4 = httpx.post("http://127.0.0.1:8080/survey", data={"name": "K", "email": "k@test.com", "rating": 5})
print("POST /survey:", r4.status_code, r4.json())

r5 = httpx.get("http://127.0.0.1:8080/survey/responses")
print("GET /survey/responses:", r5.json())

POST /log: 200 {'status': 'captured'}
GET /logs/past: [{'client_id': 'user123', 'request_type': 'login', 'timestamp': '2026-10-11T02:37:17.068477', 'seconds': 5}]
GET /survey (HTML): 
    <html><body>
        <h2>Survey</h2>
        <form action="/survey" method="post">
             ...
POST /survey: 200 {'status': 'saved', 'entry': {'name': 'K', 'email': 'k@test.com', 'rating': 5}}
GET /survey/responses: [{'name': 'K', 'email': 'k@test.com', 'rating': 5}]
